<a href="https://colab.research.google.com/github/whyaskidk91-web/saturnGPT_v2/blob/main/saturnGPT_QAchatbot_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SATURN GPT v2**

This is a basic Transformer model in GPT2 Style with 120million parameters.

BUILD:
- It uses BPE tokenizer(~16k tokens)
- AdamW
- causal multi head self-attention
- bf16
- transformer(14 pre norm blocks with 12 attention heads per block)
- context length of 768 and d_model 768
- optional dropout
- gradient clipping
- cosine learn rate
- warmup
- weight decay
- GELU MLP


DATA:
|training phase |Dataset |What got used |Size in Saturn |
|---|---|---|---|
| 1 | TinyStories V2 (GPT-4) | first ~800M characters | part of the ~1.73B pretraining tokens |
| 1 | FineWeb-Edu (sample-10BT) | streamed ~6.5B characters | the rest of the ~1.73B pretraining tokens |
| 2 | smol-smoltalk | full train split | ~412M tokens (mid-training) |
| 3 | everyday-conversations-llama3.1-2k	| ~2,250 conversations | SFT |
| 3 | Dolly-15k | 15,011 examples	 | SFT, together with the line above: ~23k training pairs |

AGENDA:
- [0],[1],..: necessarities and set up to chat
- [A],[B],..: prepare the data(once)
- [I],[II],..: training

HOW TO RUN:
- use the RUN_ME in repository to run the model in one click
- This file creates and uses files from your drive to work
- Run [A],... for loading the data.
- Run [I],... for training the model.
- Run [0],... for running the model by laoding it from your drive
- A100 suggested for computing




In [ ]:
#[0][I][A] IMPORT

import torch,re,math as m,time,os
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from google.colab import drive
DRIV = '/content/drive/MyDrive/'
drive.mount('/content/drive') #connects to google drive
dev = "cuda" if torch.cuda.is_available() else "cpu"
END = "<|endoftext|>"
pat = re.compile(r"<\|endoftext\|>|'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\w\s<]+| ?<+|\s+")
ctx = 768 #context window
print(dev,ctx)

Mounted at /content/drive
cuda 768


In [ ]:
#[-][-][B] LOAD DATASETS
# -  only do once
!wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt -O tiny_train.txt
from datasets import load_dataset
stories = open('tiny_train.txt').read(20_000_000)
fw = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
smol = load_dataset("HuggingFaceTB/smol-smoltalk",split="train")
parts,total = [stories],0
for r in fw:
  parts.append(r["text"] + END)
  total += len(r["text"])
  if total > 20_000_000: break
stories = "".join(parts)
parts,total = [],0
for r in smol:
  msgs = [x for x in r["messages"] if x["role"] in ("user","assistant")]
  for u,a in zip(msgs[0::2],msgs[1::2]):
    s = "User: " + u["content"] + "\nAssistant: " + a["content"] + END
    parts.append(s);total += len(s)
  if total > 20_000_000: break
chat = "".join(parts)
print(len(stories) / 1e6,"M chars stories |",len(chat) / 1e6,"M chars chat")

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  230MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  230MB            

... (6 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/460341 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24229 [00:00<?, ? examples/s]

40.056708 M chars stories | 20.000115 M chars chat


In [ ]:
#[1][II][C] TOKENIZER FUNCTIONS

def get_pairs(ids):
  counts = {}
  for pair in zip(ids,ids[1:]):
    if pair in counts:
      counts[pair] += 1
    else:
      counts[pair] = 1
  return counts

def merge(ids,pair,new_id):
  out = []
  i = 0
  while i < len(ids):
    if (i+1) < len(ids) and (ids[i],ids[i + 1]) == pair:
      out.append(new_id)
      i += 2
    else:
      out.append(ids[i])
      i += 1
  return out

def clean(s): return bad.sub('',s)

cache = {}
def encode_word(w):
  if w in cache: return cache[w] #every word gets encoded only once
  ids = [stoi[c] for c in w]
  while len(ids) > 1:
    pairs = get_pairs(ids)
    best = min(pairs,key = lambda p: merges.get(p, float('inf'))) #merges in order
    if best not in merges: break
    ids = merge(ids,best,merges[best])
  if len(cache) < 3_000_000: cache[w] = ids
  return ids

def encode(text):
  num_text = []
  for w in pat.findall(clean(text)): #splits text into words,punct. and spaces
    num_text += encode_word(w)
  return num_text

def decode(ids):
  return ''.join(vocab[i] for i in ids)


In [ ]:
#[-][-][D] BUILD TOKENS
# -  only do once

sample = stories + chat
cc = Counter(sample)
chars = sorted(c for c in cc if cc[c] > 50) #ignore rare chars
k0 = len(chars)
stoi = {c: a for a,c in enumerate(chars)}
bad = re.compile('[^' + re.escape(''.join(chars)) + ']')

words = { word: count for word,count in Counter(pat.findall(clean(sample))).items()
 if count >= 3 and word != END} #ignore rare words
split = {word: [stoi[c] for c in word] for word in words}

merges = {}
pairs = {}
for word,cnt in words.items():
  ids = split[word]
  for p in zip(ids,ids[1:]):
    pairs[p] = pairs.get(p,0) + cnt

for n in range(16000):
  if pairs == {}: break
  top = max(pairs,key=pairs.get)
  if pairs[top] < 1: break
  merges[top] = k0 + n
  for word,cnt in words.items():
    ids = split[word]
    if len(ids) > 1 and top[0] in ids: #skip words without the pair in it
      new_ids = merge(ids,top,k0 + n)
      if new_ids != ids:
        for p in zip(ids,ids[1:]):
          pairs[p] -= cnt
        split[word] = new_ids
        for p in zip(new_ids,new_ids[1:]):
          pairs[p] = pairs.get(p,0) + cnt
  if (n + 1) % 200 == 0: print((n + 1),end=" ")

torch.save({'characters': chars,
            'merges': merges},
           DRIV + 'tok_120M.pt')


200 400 600 800 1000 1200 1400 1600 1800 2000 2200 2400 2600 2800 3000 3200 3400 3600 3800 4000 4200 4400 4600 4800 5000 5200 5400 5600 5800 6000 6200 6400 6600 6800 7000 7200 7400 7600 7800 8000 8200 8400 8600 8800 9000 9200 9400 9600 9800 10000 10200 10400 10600 10800 11000 11200 11400 11600 11800 12000 12200 12400 12600 12800 13000 13200 13400 13600 13800 14000 14200 14400 14600 14800 15000 15200 15400 15600 15800 16000 

In [ ]:
#[2][III][E] LOAD BUILT TOKENS

loaded = torch.load(DRIV + 'tok_120M.pt',weights_only=False)
chars = loaded['characters'];merges = loaded['merges']
stoi = {c: a for a,c in enumerate(chars)}
itos = {a: c for a,c in enumerate(chars)}
bad = re.compile('[^' + re.escape(''.join(chars)) + ']')
vocab = {i: itos[i] for i in range(len(chars))}
for (a,b),new_id in merges.items():
  vocab[new_id] = vocab[a] + vocab[b]

end_id = len(chars) + len(merges)
vocab[end_id] = END #END is now one token
k = end_id + 1
cache = {END: [end_id]}
print(k,encode("Hi!" + END),decode(encode("Hello! How are you today?")))

16160 [2337, 3, 16159] Hello! How are you today?


In [ ]:
#[-][-][F] ENCODE DATA
# -  only do once
def encode_big(text,chunk=10_000_000):
  text_encode = []
  for i in range(0,len(text),chunk): #chunk to save RAM
    e = torch.tensor(encode(text[i:(i+chunk)]),dtype=torch.int16) #4x less memory
    text_encode.append(e)
    print(len(text_encode), end=" ")
  return torch.cat(text_encode)

print()
big_stories = [encode_big(open('tiny_train.txt').read(800_000_000))]
fw = load_dataset("HuggingFaceFW/fineweb-edu",name = "sample-10BT",split = "train",streaming = True)
parts,total,full = [],0,0
for r in fw:
  s = r["text"] + END
  parts.append(s);total += len(s);full += len(s)
  if total > 200_000_000:
    big_stories.append(encode_big("".join(parts)))
    parts,total = [],0
  if full > 6_500_000_000: break

if parts: big_stories.append(encode_big("".join(parts)))
big_stories = torch.cat(big_stories)
torch.save(big_stories,DRIV + 'pre_120M.pt')

parts = []
for r in smol:
  msgs = [x for x in r["messages"] if x["role"] in ("user","assistant")]
  for u,a in zip(msgs[::2],msgs[1::2]):
    parts.append("User: " + u["content"] + "\nAssistant: " + a["content"] + END)
big_chat = encode_big("".join(parts))
torch.save(big_chat,DRIV + 'mid_120M.pt')
print(len(big_chat)/1e6,"M CHAT TOKEN |",len(big_stories)/1e6,"M STORY TOKEN")


1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 1 2 3 4 5 6 7 8 9 10 11 12 1

In [ ]:
#[-][IV][G] LOAD ENCODED DATA
# -  only do when training
pre_load = torch.load(DRIV + 'pre_120M.pt')
mid_load = torch.load(DRIV + 'mid_120M.pt')
cut = int(0.99 * len(pre_load))
pre_train,pre_test = pre_load[:cut],pre_load[cut:]
cut = int(0.99 * len(mid_load))
mid_train,mid_test = mid_load[:cut],mid_load[cut:]
print(len(pre_train),len(pre_test),len(mid_train),len(mid_test))

1726356879 17437949 412399714 4165654


In [ ]:
#[-][V][-] BATCH
# -   only do when training
import torch

def batch(bD,it):
  idx = torch.randint(0,len(bD) - ctx - 1,size=(it,))
  X = (bD[idx[:,None] + torch.arange(ctx)]).long().to(dev) #Embedding needs 64bit ints
  Y = (bD[idx[:,None] + torch.arange(1,ctx + 1)]).long().to(dev)
  return X,Y

In [ ]:
#[3][VI][-] TRANSFORMER
# -    only do when training
import torch.nn as nn
import torch.nn.functional as F


d = 768
nh = 12
it = 64
block_num = 14
drp = 0.0

class MultiHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.drop = nn.Dropout(drp)
    self.hs = d // nh
    self.proj = nn.Linear(d,d)
    self.query = nn.Linear(d,d,bias=False)
    self.key = nn.Linear(d,d,bias=False)
    self.value = nn.Linear(d,d,bias=False)


  def forward(self,x):
    B,T,_ = x.shape
    q = self.query(x).view(B,T,nh,self.hs).transpose(1,2)
    kk = self.key(x).view(B,T,nh,self.hs).transpose(1,2)
    v = self.value(x).view(B,T,nh,self.hs).transpose(1,2)
    out = F.scaled_dot_product_attention(q,kk,v,is_causal = True)
    out = out.transpose(1,2).contiguous().view(B,T,d)
    return self.drop(self.proj(out))

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
    nn.Linear(d,d*4),
    nn.GELU(),
    nn.Linear(d*4,d),
    nn.Dropout(drp),
    )

  def forward(self,x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self):
    super().__init__()
    self.ln1 = nn.LayerNorm(d)
    self.att = MultiHead()
    self.ln2 = nn.LayerNorm(d)
    self.ff = FeedForward()

  def forward(self,x):
    x = x + self.att(self.ln1(x))
    x = x + self.ff(self.ln2(x))
    return x


class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.tok = nn.Embedding(k,d)
    self.pos = nn.Embedding(ctx,d)
    self.blocks = nn.Sequential(*[Block() for _ in range(block_num)])
    self.ln = nn.LayerNorm(d)
    self.out = nn.Linear(d,k)


  def forward(self,x):
    B,T = x.shape
    x = self.tok(x) + self.pos(torch.arange(T,device=dev))
    x = self.blocks(x)
    x = self.ln(x)
    return self.out(x)


In [ ]:
#[-][-][-] CLEAR CACHE
# -        only do when needed
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#[-][VII][-] CHECK BATCH
# -   only do when training
model = Net().to(dev)
opt = torch.optim.AdamW(model.parameters(),lr=6e-4,weight_decay=0.1)
print(sum(p.numel() for p in model.parameters())/1e6)
print_me,_ = batch(pre_train,it)
with torch.no_grad(),torch.autocast(dev,dtype=torch.bfloat16):
  print(model(print_me).shape)

124.627232
torch.Size([64, 768, 16160])


In [ ]:
#[-][VIII][-] TRAIN
# -  only do when training
def eval_lm(Dt,n=20):
  model.eval()
  with torch.no_grad(), torch.autocast(dev,dtype=torch.bfloat16):
    total = 0
    for _ in range(n):
      evX,evY = batch(Dt,it)
      total += F.cross_entropy(model(evX).view(-1,k),evY.view(-1)).item()
  model.train()
  return total / n


def train_lm(trainD,testD,T,lr_base,path,start=0):
  model.train()
  loss_percent = 0
  t0 = time.time()
  for i in range(start,T):
    opt.zero_grad(set_to_none = True)
    for _ in range(4):
      X,Y = batch(trainD,it)
      with torch.autocast(dev,dtype=torch.bfloat16): #bf16 is around 2x faster
        logits = model(X)
        loss = F.cross_entropy(logits.view(-1,k),Y.view(-1)) / 4
      loss_percent += loss.item()
      loss.backward()
    lr = lr_base * 0.5 * (1 + m.cos(m.pi * i / T))
    lr *= min(1,(i+1)/400) #warmup
    for g in opt.param_groups:
      g['lr'] = lr
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    opt.step()
    if (i + 1) % 250 == 0:
      print("STEP:",(i + 1),"TEST LOSS:",eval_lm(testD),"TRAIN LOSS:",
            loss_percent/250,"TIME:",(time.time() - t0) / 60,"min")
      loss_percent = 0
    if (i + 1) % 1000 == 0 or (i + 1) == T:
      torch.save({'model': model.state_dict(),
                  'opt': opt.state_dict(),
                  'step': (i + 1)
                  },path)

In [ ]:
#[-][-][-] RESUME WHEN DISCONNECTED
# -  only do when accidentally disconnected and training
ckpt = torch.load(DRIV + 'ckpt_pre_120M.pt')
model.load_state_dict(ckpt['model'])
opt.load_state_dict(ckpt['opt'])
start = ckpt['step']
print("resuming at step", start)
T = 9500
lr_base = 6e-4
path = DRIV + 'ckpt_pre_120M.pt'
train_lm(pre_train,pre_test,T,lr_base,path,start)
torch.save(model.state_dict(),DRIV + 'pre_120M_model.pt')

resuming at step 5000
STEP: 5250 TEST LOSS: 3.307103228569031 TRAIN LOSS: 3.0868759243488313 TIME: 4.756026208400726 min
STEP: 5500 TEST LOSS: 3.321671199798584 TRAIN LOSS: 3.077681774377823 TIME: 9.517742904027303 min
STEP: 5750 TEST LOSS: 3.304952096939087 TRAIN LOSS: 3.061526432514191 TIME: 14.278395171960195 min
STEP: 6000 TEST LOSS: 3.271211564540863 TRAIN LOSS: 3.049900531053543 TIME: 19.038665374120075 min
STEP: 6250 TEST LOSS: 3.278742265701294 TRAIN LOSS: 3.033589265346527 TIME: 23.851424511273702 min
STEP: 6500 TEST LOSS: 3.264573311805725 TRAIN LOSS: 3.025374596118927 TIME: 28.613404762744903 min
STEP: 6750 TEST LOSS: 3.234179937839508 TRAIN LOSS: 3.0125017538070678 TIME: 33.37598749399185 min
STEP: 7000 TEST LOSS: 3.2432594299316406 TRAIN LOSS: 3.003714677810669 TIME: 38.13887863953908 min
STEP: 7250 TEST LOSS: 3.2212499260902403 TRAIN LOSS: 2.992579509496689 TIME: 42.954712581634524 min
STEP: 7500 TEST LOSS: 3.2235320806503296 TRAIN LOSS: 2.984188960313797 TIME: 47.7202991

In [ ]:
#[-][IX][-] TRAIN PHASE 1
# -   only do when training
T = 9500
lr_base = 6e-4
path = DRIV + 'ckpt_pre_120M.pt'
train_lm(pre_train,pre_test,T,lr_base,path)
torch.save(model.state_dict(),DRIV + 'pre_120M_model.pt')

STEP: 250 TEST LOSS: 5.993780660629272 TRAIN LOSS: 6.860689592838288 TIME: 4.754943414529165 min
STEP: 500 TEST LOSS: 5.299887490272522 TRAIN LOSS: 5.4277334094047545 TIME: 9.519913427035014 min
STEP: 750 TEST LOSS: 4.696400213241577 TRAIN LOSS: 4.74778614616394 TIME: 14.280834603309632 min
STEP: 1000 TEST LOSS: 4.289893174171448 TRAIN LOSS: 4.244389393568039 TIME: 19.04118801355362 min
STEP: 1250 TEST LOSS: 4.022723722457886 TRAIN LOSS: 3.8892394344806673 TIME: 23.844595857461293 min
STEP: 1500 TEST LOSS: 3.8770556926727293 TRAIN LOSS: 3.6948753266334533 TIME: 28.60223724444707 min
STEP: 1750 TEST LOSS: 3.7716731429100037 TRAIN LOSS: 3.5748602266311647 TIME: 33.35999973615011 min
STEP: 2000 TEST LOSS: 3.6936484217643737 TRAIN LOSS: 3.4956647524833677 TIME: 38.11669944524765 min
STEP: 2250 TEST LOSS: 3.6377847671508787 TRAIN LOSS: 3.427727465867996 TIME: 42.9251258333524 min
STEP: 2500 TEST LOSS: 3.5707378149032594 TRAIN LOSS: 3.3823405976295473 TIME: 47.682487360636394 min
STEP: 2750 

KeyboardInterrupt: 

In [ ]:
#[-][X][-] LOAD DATA PHASE 3
# -       only do when train and once while loading data
import os
from datasets import load_dataset
chat_train = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="train_sft")
chat_test = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="test_sft")
dolly = load_dataset("databricks/databricks-dolly-15k",split="train")
END = "<|endoftext|>"
path = '/content/drive/MyDrive/chat_data_120M.pt'


def build(split,add_dolly):
  BX,BY,BM,pairs = [],[],[],[]
  for r in split:
    msgs = r["messages"]
    for u,a in zip(msgs[::2],msgs[1::2]):
      pairs.append((u["content"],a["content"]))
  if add_dolly:
    for r in dolly:
      u_txt = r["instruction"] + ("\n" + r["context"] if r["context"] else "")
      a_txt = r["response"]
      pairs.append((u_txt,a_txt))
  for u,a in pairs:
    p = encode("User: " + u + "\nAssistant:")
    q = encode(" " + a + END)
    ids = p + q
    mask = [0] * len(p) + [1] * len(q) #only answer tokens count
    if len(ids) > (ctx + 1): continue
    pad = ctx + 1 - len(ids) #padding
    ids += [0] * pad
    mask += [0] * pad
    BX.append(ids[:-1]);BY.append(ids[1:]);BM.append(mask[1:])
  return (torch.tensor(BX,device=dev),torch.tensor(BY,device=dev),
          torch.tensor(BM,device=dev,dtype=torch.float))

if os.path.exists(path):
  CX,CY,CM,TX,TY,TM = torch.load(path)
else:
  CX,CY,CM = build(chat_train,True)
  TX,TY,TM = build(chat_test,False)
  torch.save((CX,CY,CM,TX,TY,TM),path)
print(CX.shape,TX.shape)

README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

data/train_sft-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.05MB            

data/train_sft-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test_sft-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  124kB            

data/test_sft-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train_sft split:   0%|          | 0/2260 [00:00<?, ? examples/s]

... (2 outputs left out)

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

torch.Size([23173, 768]) torch.Size([455, 768])


In [ ]:
#[-][XI][-] FINE TUNE BATCH
# -     only run when training
def chat_batch(cbX,cbY,cbM,it):
  idx = torch.randint(0,len(cbX),(it,),device=dev)
  return cbX[idx],cbY[idx],cbM[idx]

def masked_loss(logits,mY,mM): #loss of only assisstants answers
  l = F.cross_entropy(logits.view(-1,k),mY.view(-1),reduction='none')
  return (l * mM.view(-1)).sum() / mM.sum()

In [ ]:
#[-][XII][-] TRAIN PHASE 2
# -   only run when training
model = Net().to(dev)
model.load_state_dict(torch.load(DRIV + 'pre_120M_model.pt'))
opt = torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=0.1)
T = 3000
lr_base = 2e-4
path = DRIV + 'ckpt_mid_120M.pt'
train_lm(mid_train,mid_test,T,lr_base,path)
torch.save(model.state_dict(),DRIV + 'mid_120M_model.pt')

STEP: 250 TEST LOSS: 2.3027568817138673 TRAIN LOSS: 2.5550668153762817 TIME: 4.78663203716278 min
STEP: 500 TEST LOSS: 2.1387563228607176 TRAIN LOSS: 2.214396123766899 TIME: 9.553925955295563 min
STEP: 750 TEST LOSS: 2.074665880203247 TRAIN LOSS: 2.0975271906852724 TIME: 14.32106248140335 min
STEP: 1000 TEST LOSS: 2.023404359817505 TRAIN LOSS: 2.023955904006958 TIME: 19.088263726234437 min
STEP: 1250 TEST LOSS: 1.9717424154281615 TRAIN LOSS: 1.9753964599370957 TIME: 23.912333393096922 min
STEP: 1500 TEST LOSS: 1.956349539756775 TRAIN LOSS: 1.943498951435089 TIME: 28.680772932370505 min
STEP: 1750 TEST LOSS: 1.9146973609924316 TRAIN LOSS: 1.91196290910244 TIME: 33.4486243724823 min
STEP: 2000 TEST LOSS: 1.892783373594284 TRAIN LOSS: 1.891991027355194 TIME: 38.21674424409866 min
STEP: 2250 TEST LOSS: 1.891855639219284 TRAIN LOSS: 1.8690819877386093 TIME: 43.041663821538286 min
STEP: 2500 TEST LOSS: 1.8866364121437074 TRAIN LOSS: 1.8595628325939177 TIME: 47.80911372502645 min
STEP: 2750 T

In [ ]:
#[-][XIII][-] TRAIN PHASE 3
# -  only do when training
import math as m
model = Net().to(dev)
model.load_state_dict(torch.load(DRIV + 'mid_120M_model.pt'))
opt = torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=0.01)
T,lr_base,sft_it = 2000,5e-5,32
for i in range(T):
  model.train()
  X,Y,M = chat_batch(CX,CY,CM,sft_it)
  with torch.autocast(dev,dtype=torch.bfloat16):
    loss = masked_loss(model(X),Y,M)
  lr = lr_base * 0.5 * (1 + m.cos(m.pi * i / T))
  for g in opt.param_groups: g['lr'] = lr
  opt.zero_grad();loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
  opt.step()
  if (i + 1) % 250 == 0:
    model.eval()
    with torch.no_grad():
      Xt,Yt,Mt = chat_batch(TX,TY,TM,sft_it*4)
      tl = masked_loss(model(Xt),Yt,Mt).item()
    print(f"step {i + 1} train {loss.item():.3f} test {tl:.3f}")
    torch.save(model.state_dict(),DRIV + 'chat_ckpt_120M.pt')
torch.save(model.state_dict(),DRIV + 'chat_model_120M.pt')


step 250 train 2.544 test 1.755
step 500 train 1.771 test 1.676
step 750 train 1.988 test 1.857
step 1000 train 2.141 test 1.838
step 1250 train 1.796 test 1.882
step 1500 train 2.034 test 1.881
step 1750 train 1.744 test 1.794
step 2000 train 1.661 test 1.905


In [ ]:
#[4][XIV][-] LOAD FULL MODEL

model = Net().to(dev)
model.load_state_dict(torch.load(DRIV + 'chat_model_120M.pt',map_location=torch.device(dev)))

<All keys matched successfully>

In [ ]:
#[5][XV][-] RUN FULL MODEL

model.eval()
while True:
  Temp = 0.7
  msg = input("You: ")
  if msg == "q": break
  ids = encode("User: " + clean(msg) + "\nAssistant:")
  out = []
  with torch.no_grad():
    for _ in range(200):
      x = torch.tensor([ids[-ctx:]],device=dev)
      probs = F.softmax(model(x)[:,-1,:] / Temp,dim=1)[0]
      t = torch.multinomial(probs,1).item()
      if t == end_id: break
      ids.append(t);out.append(t)
  print("Bot:",decode(out).strip())